# Analyse movement bouts
The goal here is to check the shape of the data, specifically if we're able to run the analysis that we want.
## Aim
En substance, we would like to:
* Extract pre-motor activity in moments that the animal is otherwise (relatively) immobile, both in tracking and playback
* Guess features of movement from this activity, such as direction, speed, distance to final position, etc.
* This could be done via dimensionality reduction, direct regression, other sophisticated tech.
## Limits
The list of answerable questions is large, but we need to start by figuring out if the data is plentyful enough

## Parameters
We will select:

**Immobility**

Absolute position velocity stays continuously below 0.02 position units/s for at least 1.2 s.

**Movement onset**

The first sample following a qualifying immobility period. We apply the same definition in tracking and playback.

**Movement endpoint and final position**

Provisionally, the endpoint is the start of the next qualifying immobility period, and final position is Position at that sample. If none follows before the session ends, the endpoint stays missing and the onset remains available for pre-motion analysis.

## Saved info:
We record:
* Identification data: animal, headstage, source session
* Condition: i.e., playback or tracking (0/1, as is already in feature). Set at movement onset (to avoid complicated mixed case)
* onset bin/time
* end bin/time
* Initial position
* Final position
* Total movement
* Direction (initial, to avoid def. problems if it changes, so what matters is the first steps. might be redefined later)
* Speed (avg speed, I guess?)


# 1. Load data

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display

# Locate the shared data folder from the workspace, repo, or notebooks folder.
data_dir = next(
    (parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Cannot locate the workspace data directory.")

feature_paths = sorted(data_dir.glob("*_AC_playback_0.005_feature_ss"))
if not feature_paths:
    raise FileNotFoundError(f"No AC feature files found in {data_dir}")

dt = 0.005  # seconds per bin
features = []
session_rows = []
for path in feature_paths:
    animal, _, headstage, region, *_ = path.name.split("_")
    with path.open("rb") as stream:
        collection_features = pickle.load(stream)

    for session, session_features in enumerate(collection_features):
        features.append(session_features)
        session_rows.append({
            "animal": animal,
            "headstage": int(headstage),
            "region": region,
            "session": session,
            "feature_file": path.name,
            "n_bins": len(session_features),
            "duration_s": len(session_features) * dt,
            "tracking_s": session_features["Condition"].eq(0).sum() * dt,
            "playback_s": session_features["Condition"].eq(1).sum() * dt,
        })
    del collection_features, session_features

# features[s][column] corresponds to session_info.loc[s].
# Session entries count each headstage separately.
session_info = pd.DataFrame(session_rows)
print(f"Loaded {len(features)} AC session entries from {len(feature_paths)} collections.")
display(session_info)


# 2. Do itttt

In [ ]:
# Enter parameters for analysis
DT = 0.005
# Speed_x uses position differences * 100. For 5 ms bins, * 2 converts to position units/s.
SPEED_X_FACTOR = 2.0

VELOCITY_THRESHOLD = 0.02
MIN_IMMOBILITY_S = 1.2

In [ ]:
# Run with movement_bouts/ as the working directory.
from utils import detect_movement_bouts

movement_tables = []
immobility_tables = []

for idx in range(len(features)):
    info = session_info.loc[idx]
    session_movement_bouts, session_immobility_bouts = detect_movement_bouts(
        features[idx],
        info,
        dt=DT,
        speed_x_factor=SPEED_X_FACTOR,
        velocity_threshold=VELOCITY_THRESHOLD,
        min_immobility_s=MIN_IMMOBILITY_S,
    )

    # Keep a direct reference to features[idx] in both combined tables.
    session_movement_bouts["idx"] = idx
    session_immobility_bouts["idx"] = idx
    for column in ("animal", "headstage", "session"):
        session_immobility_bouts[column] = info[column]

    movement_tables.append(session_movement_bouts)
    immobility_tables.append(session_immobility_bouts)

movement_bouts = pd.concat(movement_tables, ignore_index=True)
immobility_bouts = pd.concat(immobility_tables, ignore_index=True)

print(f"Processed {len(features)} session entries.")
print(f"Movement bouts: {len(movement_bouts)}")
print(f"Immobility segments: {len(immobility_bouts)}")
display(movement_bouts)


# 3. Plots, verifications, analysis
Now that we extracted everything, we can plot stuff to verify we did well, analyse the shape of our data, see what's possible to do.

In [ ]:
# Browse the full session with a moving time window.
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Patch

# Select a session to inspect from the combined tables.
idx = 0
position = features[idx]["Position"].to_numpy()
speed = features[idx]["Speed_x"].to_numpy() * SPEED_X_FACTOR
time = np.arange(len(position)) * DT
plot_immobility_bouts = immobility_bouts.loc[immobility_bouts["idx"] == idx]

window_s = 100  # Width of the visible window, in seconds.
session_duration_s = len(position) * DT
max_start_s = max(0.0, session_duration_s - window_s)

start_slider = widgets.FloatSlider(
    value=min(1000.0, max_start_s),
    min=0.0,
    max=max_start_s,
    step=DT,
    description="Start (s)",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)


def plot_immobility_window(start_s):
    stop_s = min(start_s + window_s, session_duration_s)
    visible = (time >= start_s) & (time < stop_s)
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(time[visible], position[visible], color="slategray", linewidth=1)
    axes[0].set_ylabel("Position")
    axes[1].plot(time[visible], speed[visible], color="slategray", linewidth=1)
    axes[1].axhline(VELOCITY_THRESHOLD, color="purple", linestyle="--",
                    label="Immobility speed threshold")
    axes[1].set_ylabel("Speed (position units/s)")
    axes[1].set_xlabel("Time (s)")

    for bout in plot_immobility_bouts.itertuples(index=False):
        bout_start_s = bout.start_bin * DT
        bout_stop_s = bout.stop_bin * DT  # stop_bin is excluded
        if bout_start_s < stop_s and bout_stop_s > start_s:
            color = "tab:blue" if bout.qualifies else "tab:orange"
            for ax in axes:
                ax.axvspan(max(bout_start_s, start_s), min(bout_stop_s, stop_s),
                           color=color, alpha=0.25, linewidth=0)

    axes[0].legend(handles=[
        Patch(facecolor="tab:blue", alpha=0.25,
              label=f"Immobile for at least {MIN_IMMOBILITY_S:g} s"),
        Patch(facecolor="tab:orange", alpha=0.25, label="Short immobility"),
    ], loc="upper right")
    axes[1].legend(loc="upper right")
    axes[1].set_xlim(start_s, stop_s)
    fig.suptitle(f"Session index {idx}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


bout_plot = widgets.interactive_output(
    plot_immobility_window, {"start_s": start_slider}
)
display(start_slider, bout_plot)


In [ ]:
import matplotlib.pyplot as plt

bout_counts = (
    movement_bouts["condition"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Tracking", "Playback"], bout_counts.to_numpy(), color=["red", "black"])
ax.bar_label(bars, padding=3)
ax.set_ylabel("Number of movement bouts")
ax.set_ylim(0, max(1, bout_counts.max()) * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

POSITION_MIDPOINT = 14.0  # Midpoint of the reconstructed 0-28 position range.

# Use coordinate direction until the physical left/right mapping is confirmed.
direction_labels = ["Decreasing position", "Increasing position"]
directions = [-1, 1]
conditions = [(0, "Tracking", "red"), (1, "Playback", "black")]
x = np.arange(len(directions))
bar_width = 0.35

panels = [
    ("All starting positions", movement_bouts["initial_position"].notna()),
    (f"Starts below {POSITION_MIDPOINT:g}",
     movement_bouts["initial_position"] < POSITION_MIDPOINT),
    (f"Starts at or above {POSITION_MIDPOINT:g}",
     movement_bouts["initial_position"] >= POSITION_MIDPOINT),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
largest_count = 0
for ax, (title, position_mask) in zip(axes, panels):
    for condition_index, (condition, label, color) in enumerate(conditions):
        counts = (
            movement_bouts.loc[
                position_mask & movement_bouts["condition"].eq(condition),
                "initial_direction",
            ]
            .value_counts()
            .reindex(directions, fill_value=0)
        )
        offset = (condition_index - 0.5) * bar_width
        bars = ax.bar(x + offset, counts.to_numpy(), width=bar_width,
                      color=color, label=label)
        ax.bar_label(bars, padding=3)
        largest_count = max(largest_count, counts.max())
    ax.set_title(title)
    ax.set_xticks(x, direction_labels)

axes[0].set_ylabel("Number of movement bouts")
axes[0].set_ylim(0, max(1, largest_count) * 1.15)
axes[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preserve the histogram counts, with a point at each one-unit bin centre.
position_bins = np.arange(0, 29, 1)
position_centres = (position_bins[:-1] + position_bins[1:]) / 2
conditions = [(0, "Tracking", "red"), (1, "Playback", "black")]
direction_panels = [(-1, "Decreasing position"), (1, "Increasing position")]

with sns.axes_style("ticks"), sns.plotting_context("notebook"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
    for ax, (direction, title) in zip(axes, direction_panels):
        for condition, label, color in conditions:
            starting_positions = movement_bouts.loc[
                movement_bouts["initial_direction"].eq(direction)
                & movement_bouts["condition"].eq(condition),
                "initial_position",
            ].dropna().to_numpy()
            counts, _ = np.histogram(starting_positions, bins=position_bins)
            ax.plot(
                position_centres, counts,
                color=color, linewidth=2, marker="o", markersize=5,
                markerfacecolor="white", markeredgewidth=1.5, label=label,
            )
        ax.axvline(14, color="0.7", linestyle="--", linewidth=1, zorder=0)
        ax.set_title(title, pad=12)
        ax.set_xlabel("Starting position")
        ax.set_xlim(0, 28)
        ax.set_ylim(bottom=0,top=500)
        ax.set_xticks(np.arange(0, 29, 4))
        ax.grid(axis="y", color="0.92", linewidth=0.8)
        ax.set_axisbelow(True)

    axes[0].set_ylabel("Movement bouts per position unit")
    axes[0].legend(frameon=False)
    axes[1].tick_params(axis="y",labelleft=True)
    sns.despine(fig=fig)
    fig.suptitle(f"All {len(session_info)} session entries pooled", y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# Compare tracking and playback within each session.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

position_bins = np.arange(0, 29, 1)
position_centres = (position_bins[:-1] + position_bins[1:]) / 2
conditions = [(0, "Tracking", "red"), (1, "Playback", "black")]
direction_panels = [(-1, "Decreasing position"), (1, "Increasing position")]

session_selector = widgets.Dropdown(
    options=[
        (f"idx {idx} | {info['animal']} | hs {info['headstage']} | session {info['session']}", idx)
        for idx, info in session_info.iterrows()
    ],
    description="Session",
    layout=widgets.Layout(width="650px"),
)


def plot_session_directions(idx):
    info = session_info.loc[idx]
    session_bouts = movement_bouts.loc[movement_bouts["idx"].eq(idx)]
    with sns.axes_style("ticks"), sns.plotting_context("notebook"):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
        for ax, (direction, title) in zip(axes, direction_panels):
            for condition, label, color in conditions:
                starting_positions = session_bouts.loc[
                    session_bouts["initial_direction"].eq(direction)
                    & session_bouts["condition"].eq(condition),
                    "initial_position",
                ].dropna().to_numpy()
                counts, _ = np.histogram(starting_positions, bins=position_bins)
                ax.plot(
                    position_centres, counts,
                    color=color, linewidth=2, marker="o", markersize=5,
                    markerfacecolor="white", markeredgewidth=1.5, label=label,
                )
            ax.axvline(14, color="0.7", linestyle="--", linewidth=1, zorder=0)
            ax.set_title(title, pad=12)
            ax.set_xlabel("Starting position")
            ax.set_xlim(0, 28)
            ax.set_ylim(bottom=0)
            ax.set_xticks(np.arange(0, 29, 4))
            ax.grid(axis="y", color="0.92", linewidth=0.8)
            ax.set_axisbelow(True)

        axes[0].set_ylabel("Movement bouts per position unit")
        axes[0].legend(frameon=False)

        max_count = max(
        line.get_ydata().max()
        for ax in axes
        for line in ax.lines[:2]  # Tracking and playback curves
        )
        axes[0].set_ylim(0, max(1, max_count) * 1.05)

        sns.despine(fig=fig)
        fig.suptitle(
            f"{info['animal']} | headstage {info['headstage']} | "
            f"session {info['session']} | idx {idx}", y=1.02,
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)


session_direction_plot = widgets.interactive_output(
    plot_session_directions, {"idx": session_selector}
)
display(session_selector, session_direction_plot)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Show each animal/session once. Prefer hs0, retaining hs1-only sessions.
plot_sessions = (
    session_info.sort_values(["animal", "session", "headstage"])
    .drop_duplicates(subset=["animal", "session"], keep="first")
)

# Each row is one session's starting-position histogram, combining directions.
position_bins = np.arange(0, 29, 1)
position_centres = (position_bins[:-1] + position_bins[1:]) / 2
position_histograms = {}

for condition in (0, 1):
    counts = pd.DataFrame(
        0, index=plot_sessions.index, columns=position_centres, dtype=int
    )
    for idx in plot_sessions.index:
        starting_positions = movement_bouts.loc[
            movement_bouts["idx"].eq(idx)
            & movement_bouts["condition"].eq(condition),
            "initial_position",
        ].dropna().to_numpy()
        counts.loc[idx] = np.histogram(starting_positions, bins=position_bins)[0]
    position_histograms[condition] = counts

# Keep session order and count scale identical across the two conditions.
max_count = max(1, *(table.to_numpy().max() for table in position_histograms.values()))
session_labels = [
    f"{idx} | {info['animal']} hs{info['headstage']} s{info['session']}"
    for idx, info in plot_sessions.iterrows()
]

with sns.axes_style("white"):
    fig, axes = plt.subplots(
        1, 2, figsize=(14, max(6, len(plot_sessions) * 0.23)), sharey=True
    )
    for ax, condition, title, cmap in zip(
        axes, (0, 1), ("Tracking", "Playback"), ("Reds", "Greys")
    ):
        sns.heatmap(
            position_histograms[condition], ax=ax, cmap=cmap,
            vmin=0, vmax=max_count,
            xticklabels=False, yticklabels=session_labels,
            cbar_kws={"label": "Number of movement bouts", "shrink": 0.5},
        )
        ax.axvline(14, color="0.6", linestyle="--", linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("Starting-position bin")
        ax.set_xticks(np.arange(0, 28, 4) + 0.5)
        ax.set_xticklabels([f"{edge}-{edge + 1}" for edge in range(0, 28, 4)])
        ax.tick_params(axis="y", labelsize=8, rotation=0)

    axes[0].set_ylabel("Session entry (idx | animal, headstage, session)")
    axes[1].set_ylabel("")
    axes[1].tick_params(axis="y", labelleft=False)
    fig.suptitle(f"Movement-bout starting positions | {len(plot_sessions)} unique sessions", y=1.005)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Show each animal/session once. Prefer hs0, retaining hs1-only sessions.
plot_sessions = (
    session_info.sort_values(["animal", "session", "headstage"])
    .drop_duplicates(subset=["animal", "session"], keep="first")
)

# Each row is one session's starting-position histogram, combining directions.
position_bins = np.arange(0, 29, 1)
position_centres = (position_bins[:-1] + position_bins[1:]) / 2
position_histograms = {}

for condition in (0, 1):
    counts = pd.DataFrame(
        0, index=plot_sessions.index, columns=position_centres, dtype=int
    )
    for idx in plot_sessions.index:
        starting_positions = movement_bouts.loc[
            movement_bouts["idx"].eq(idx)
            & movement_bouts["condition"].eq(condition),
            "initial_position",
        ].dropna().to_numpy()
        counts.loc[idx] = np.histogram(starting_positions, bins=position_bins)[0]
    position_histograms[condition] = counts

# Keep session order and count scale identical across the two conditions.
max_count = max(1, *(table.to_numpy().max() for table in position_histograms.values()))
session_labels = [
    f"{idx} | {info['animal']} hs{info['headstage']} s{info['session']}"
    for idx, info in plot_sessions.iterrows()
]

# Direction totals use the same selected session entries as the heatmaps.
direction_counts = {}
for condition in (0, 1):
    direction_counts[condition] = (
        movement_bouts.loc[movement_bouts["condition"].eq(condition)]
        .groupby(["idx", "initial_direction"])
        .size()
        .unstack("initial_direction", fill_value=0)
        .reindex(index=plot_sessions.index, columns=[-1, 1], fill_value=0)
    )

max_total = max(1, *(table.to_numpy().max() for table in direction_counts.values()))
row_centres = np.arange(len(plot_sessions)) + 0.5

with sns.axes_style("white"):
    fig = plt.figure(
        figsize=(18, max(7, len(plot_sessions) * 0.28)), layout="constrained"
    )
    grid = fig.add_gridspec(
        2, 4, width_ratios=[3, 1.5, 3, 1.5], height_ratios=[1, 0.015]
    )
    first_heatmap = None
    for panel, condition, title, cmap, color in zip(
        (0, 2), (0, 1), ("Tracking", "Playback"), ("Reds", "Greys"), ("red", "black")
    ):
        heatmap_ax = fig.add_subplot(grid[0, panel], sharey=first_heatmap)
        if first_heatmap is None:
            first_heatmap = heatmap_ax
        totals_ax = fig.add_subplot(grid[0, panel + 1], sharey=first_heatmap)
        colorbar_ax = fig.add_subplot(grid[1, panel])

        sns.heatmap(
            position_histograms[condition], ax=heatmap_ax, cmap=cmap,
            vmin=0, vmax=max_count,
            xticklabels=False, yticklabels=session_labels,
            cbar_ax=colorbar_ax,
            cbar_kws={"orientation": "horizontal", "label": "Bouts per position unit"},
        )
        heatmap_ax.set_title(title)
        heatmap_ax.set_xlabel("Starting-position bin")
        heatmap_ax.set_xticks(np.arange(0, 28, 4) + 0.5)
        heatmap_ax.set_xticklabels([f"{edge}-{edge + 1}" for edge in range(0, 28, 4)])
        heatmap_ax.axvline(14, color="0.6", linestyle="--", linewidth=0.8)
        heatmap_ax.tick_params(axis="y", labelsize=8, rotation=0, labelleft=panel == 0)
        heatmap_ax.set_ylabel("")

        counts = direction_counts[condition]
        totals_ax.barh(
            row_centres - 0.18, counts[-1], height=0.32,
            facecolor="none", edgecolor=color, linewidth=1, label="Decreasing (-)",
        )
        totals_ax.barh(
            row_centres + 0.18, counts[1], height=0.32,
            color=color, label="Increasing (+)",
        )
        totals_ax.set_xlim(0, max_total * 1.1)
        totals_ax.set_ylim(len(plot_sessions), 0)
        totals_ax.set_xlabel("Total bouts")
        totals_ax.set_title("Initial direction")
        totals_ax.tick_params(axis="y", left=False, labelleft=False)
        totals_ax.grid(axis="x", color="0.92")
        totals_ax.set_axisbelow(True)
        totals_ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.08),
                         fontsize=8)
        sns.despine(ax=totals_ax, left=True)

    first_heatmap.set_ylabel("Session entry (idx | animal, headstage, session)")
    fig.suptitle(f"Starting positions and direction totals | {len(plot_sessions)} unique sessions")
    plt.show()
